## tl;dr
77本のカーブの再取得、公開136商品と非公開24商品の分離、コスパの候補集合を監査する。

## Context & Methods
### Key Assumptions
固定提出物のAPI換算開発費と誤差を比較。隠し条件ごとに再較正。定額請求や本番利用費ではない。グラフは間引き、RMSEは全格子。

## Data
元データと再実行方法はcapture_curves.py、計算はreport_extensions.py。監査結果のパスを以下で参照。

In [1]:
import json, math
from pathlib import Path
ROOT = Path.cwd()
if not (ROOT / 'evaluator').exists(): ROOT = ROOT.parents[1]
BASE = ROOT / 'analysis/feedback-round-01-final-20260905'
quality = json.loads((BASE/'curve_shape_quality.json').read_text())
capture = json.loads((BASE/'curve_shapes/capture_audit.json').read_text())
assert len(capture['runs']) == 77 and capture['candidates_unchanged']
assert quality['public_unique_instruments'] == 136
assert quality['holdout_instruments'] == 24 and quality['overlapping_instruments'] == 0
print('PASS: 77 retained curves; 136 public + 24 disjoint holdout instruments')


PASS: 77 retained curves; 136 public + 24 disjoint holdout instruments


## Results

In [2]:
cost = json.loads((BASE/'cost_performance.json').read_text())
rows = cost['rows']
def pareto(error):
    return [r['model'] for r in rows if not any(q['cost'] <= r['cost'] and q[error] <= r[error] and (q['cost'] < r['cost'] or q[error] < r[error]) for q in rows)]
assert pareto('test_bp') == cost['test_frontier']
assert pareto('main_bp') == cost['main_frontier']
assert math.isclose(sum(r['extra_cost'] for r in rows), 66.33494834)
print('Test frontier:', pareto('test_bp'))
print('Main frontier:', pareto('main_bp'))
print('Additional cost USD:', round(sum(r['extra_cost'] for r in rows), 2))


Test frontier: ['Luna', 'Fable']
Main frontier: ['Luna', 'Opus', 'Fable']
Additional cost USD: 66.33


## Takeaways
コスト×テスト平均誤差ではLuna/Fable、主データ精度ではOpusも候補。1回の実行結果であり一般能力順位ではない。